In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IEEE.prepare_IEEE import process_all_IEEE_files
from common.moment_embeddings import get_embeddings_in_batches
from momentfm import MOMENTPipeline

/home/mdereje/Projects/mcsa_tsfm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large",
    model_kwargs={'task_name': 'embedding'}, # We are loading the model in `embedding` mode
)
model.init()
model = model.to(device).float()

/home/mdereje/Projects/mcsa_tsfm/.venv/lib/python3.11/site-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


In [ ]:
WINDOW_SIZE = 1024
OVERLAP = int(WINDOW_SIZE/2)
MAX_WINDOWS = 200  # windows kept per signal per recording (None keeps all: 1,641 per recording for a full export at 1024/512, which makes very large DataFrames)

all_processed_windows_a, all_processed_windows_b, all_processed_windows_c = process_all_IEEE_files(WINDOW_SIZE, OVERLAP, MAX_WINDOWS)

print(f"Total processed windows for 'isa': {len(all_processed_windows_a)}")
print(f"Total processed windows for 'isb': {len(all_processed_windows_b)}")
print(f"Total processed windows for 'isc': {len(all_processed_windows_c)}")

print(all_processed_windows_a[0])

Found 400 files


Processing files: 100%|██████████| 400/400 [00:01<00:00, 235.50it/s]

Total processed windows for 'isa': 2400
Total processed windows for 'isb': 2400
Total processed windows for 'isc': 2400
{'data': [1.3270835, 1.3520835, 1.3666665, 1.375, 1.3895835, 1.397917, 1.4125, 1.40625, 1.4125, 1.41875, 1.425, 1.4229165, 1.43125, 1.4375, 1.4458335, 1.44375, 1.4562499999999998, 1.4791665, 1.483333, 1.491667, 1.50625, 1.5166665, 1.525, 1.529167, 1.529167, 1.529167, 1.5291665, 1.5229165, 1.5125, 1.5020829999999998, 1.5020829999999998, 1.4854165, 1.483333, 1.4625, 1.4604165, 1.4520835, 1.4458335, 1.441667, 1.433333, 1.4270835000000002, 1.4270835000000002, 1.4229165, 1.427083, 1.4145835, 1.4041665, 1.4020834999999998, 1.395833, 1.3875, 1.3875, 1.389583, 1.3833335, 1.3770829999999998, 1.36875, 1.3854164999999998, 1.3708335, 1.370833, 1.3625, 1.3541665, 1.345833, 1.3479165000000002, 1.3395830000000002, 1.3333335, 1.3333335, 1.3333335, 1.3375, 1.320833, 1.320833, 1.3145829999999998, 1.30625, 1.2916665, 1.2916665, 1.2708335, 1.2645835, 1.2520835, 1.2520835, 1.24375, 1.2354

In [ ]:
batch_size = 256 # You can adjust this based on your GPU memory
combined_embeddings_a, combined_metadata_a = get_embeddings_in_batches(all_processed_windows_a, model, batch_size=batch_size, device=device)
combined_embeddings_b, combined_metadata_b = get_embeddings_in_batches(all_processed_windows_b, model, batch_size=batch_size, device=device)
combined_embeddings_c, combined_metadata_c = get_embeddings_in_batches(all_processed_windows_c, model, batch_size=batch_size, device=device)

#df_embeddings_b = 
#df_embeddings_c = 

Generating embeddings in batches: 100%|██████████| 10/10 [00:11<00:00,  1.17s/it]


: 

In [ ]:
def windows_to_dataframe(processed_windows):
    rows = []

    for window in processed_windows:
        row = window["metadata"].copy()

        for i, value in enumerate(window["data"]):
            row[f"sample_{i}"] = value

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Step 2: Convert windows to DataFrames
df_a = windows_to_dataframe(all_processed_windows_a)
df_b = windows_to_dataframe(all_processed_windows_b)
df_c = windows_to_dataframe(all_processed_windows_c)

# To get time we just have to divide the sample number by the sampling frequency (50 kHz here)
for df in (df_a, df_b, df_c):
    parts = df["file_name"].str.extract(r"^(?P<condition>\w+?)_torque(?P<torque>\d+)_(?P<repetition>\d+)")
    loc = df.columns.get_loc("sample_0")
    df.insert(loc, "condition", parts["condition"])
    df.insert(loc + 1, "torque", parts["torque"].astype(int) / 10)
    df.insert(loc + 1, "repetition", parts["repetition"].astype(int)) 



In [ ]:
df_a

In [ ]:
df_a.tail(10)

In [ ]:
df_a.shape